# Self-Supervised Learning and Imaging Foundation Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week06_self_supervised_learning.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 6 · Reference notebook (ungraded)**

This notebook follows the Week 6 lecture and its three clips, with the same two running examples:
- a **single-lead ECG**, 10 seconds long, made by the same synthetic generator that draws the ECGs in the clips;
- **H&E tiles** of colorectal tissue from Kather et al. 2016, the same open dataset as the tiles in the clips.

We pretrain small encoders **without labels**, then measure what they learned with a **linear probe** and with
**fine-tuning**, using only a few labels.

| Section | Clip | What you will see |
|---|---|---|
| 1 | Masked reconstruction: learning by filling in gaps | 10 one-second ECG patches, 6 hidden; an H&E tile with 12 of 16 patches hidden |
| 2 | Contrastive learning: two views, one embedding | Augmented views, the InfoNCE loss with the clip's numbers, a small contrastive run |
| 3 | Linear probing: what did the encoder learn? | A frozen encoder, a 257-weight linear head, AF versus normal and tumor versus normal mucosa |
| 4 | Optional | Features from a pretrained imaging foundation model (DINOv2) |

**How to run it**
- **Google Colab:** *Runtime > Run all*. A CPU is enough: the whole notebook takes about 12 minutes, including downloads. A GPU makes it faster.
- **HiPerGator or your own computer:** a Jupyter kernel with PyTorch. Set `DATA_DIR` in the setup cell to a folder on your
  group's `/blue` storage.
- The first run downloads the Kather tiles (about 260 MB) and, in the optional section 4, DINOv2-small (about 90 MB).

**Data.** The ECGs are synthetic, generated in code. The H&E tiles are from Kather JN et al., *Multi-class texture analysis in
colorectal cancer histology*, Scientific Reports 6, 27988 (2016), [doi:10.5281/zenodo.53169](https://doi.org/10.5281/zenodo.53169),
licensed CC BY 4.0: 5,000 tiles of 150 × 150 pixels in 8 tissue classes, from an open, de-identified dataset.

**The models are deliberately tiny** so they train on a CPU in minutes. The numbers you get show the ideas, not the accuracy
of real foundation models, which use the same recipes on millions of recordings or slides.

## 0. Setup

In [ ]:
import os, glob, copy, time, zipfile, urllib.request        # standard library: files, copies, timing, download
import numpy as np                                            # arrays for the ECG generator
import torch                                                  # tensors and autograd
import torch.nn as nn                                         # layers
import torch.nn.functional as F                               # losses and resizing
import matplotlib.pyplot as plt                               # plots
from PIL import Image                                         # reads the TIFF tiles

DATA_DIR = os.path.expanduser("~/bme6938_data")               # on HiPerGator, use a folder on /blue
os.makedirs(DATA_DIR, exist_ok=True)                          # create it if needed
torch.manual_seed(0); np.random.seed(0)                       # same random numbers on every run
print("PyTorch", torch.__version__)                           # check the install

## 1. Masked reconstruction: learning by filling in gaps  *(clip 1)*

### 1.1 Start with an ECG recording that has no label
The function below is the ECG generator used to draw the clips. Each heartbeat is a sum of five Gaussian bumps: the P wave,
the Q, R, and S waves, and the T wave. With `af=True` the beats come at irregular intervals, there is no P wave, and a small
fibrillatory wave rides on the baseline, as in atrial fibrillation (AF). We sample it at 250 samples per second.

In [ ]:
FS = 250                                                      # samples per second
DUR = 10                                                      # seconds per recording, as in the clip
t_axis = np.arange(FS * DUR) / FS                             # time of every sample, in seconds

def ecg(hr=72, af=False, seed=0, dur=DUR):
    # Single-lead synthetic ECG of `dur` seconds: the generator from the clips (lectures/_shared/kit.py).
    rng = np.random.default_rng(seed)                         # random source for this recording
    beats = [rng.uniform(0.25, 0.55)]                         # time of the first R peak
    while beats[-1] < dur + 1:                                # add beats until past the end
        gap = rng.uniform(0.42, 0.95) if af else 60 / hr * rng.uniform(0.97, 1.03)   # AF: irregular gaps; sinus: regular
        beats.append(beats[-1] + gap)
    t = np.arange(FS * dur) / FS                              # sample times
    out = np.zeros_like(t)                                    # the signal, built up beat by beat
    bump = lambda c, a, w: a * np.exp(-0.5 * ((t - c) / w) ** 2)   # one Gaussian wave: center, amplitude, width
    for c in beats:
        if not af: out += bump(c - 0.17, 0.13, 0.025)         # P wave (absent in AF)
        out += bump(c - 0.03, -0.12, 0.009) + bump(c, 1.0, 0.011) + bump(c + 0.035, -0.28, 0.011)   # Q, R, S
        out += bump(c + 0.28, 0.27, 0.045)                    # T wave
    if af:                                                    # fibrillatory baseline in AF
        ph = rng.uniform(0, 6.28, 2)
        out += 0.045 * np.sin(2 * np.pi * 6.3 * t + ph[0]) + 0.03 * np.sin(2 * np.pi * 8.9 * t + ph[1])
    return out

x = ecg(hr=72, seed=3)                                        # the recording drawn in clip 1
plt.figure(figsize=(12, 2)); plt.plot(t_axis, x, lw=1)        # plot it
plt.title("single lead, 10 s, no diagnosis"); plt.xlabel("seconds"); plt.show()

To pretrain, we need many recordings. We generate 3,000 for pretraining and 1,000 for testing. Half are AF and half are
normal sinus rhythm, with random heart rates, amplitudes, noise, and baseline wander. **Pretraining never sees the labels**
`y`: they are kept only for the linear probe in section 3.

In [ ]:
def make_ecgs(n, seed):
    # n noisy recordings; returns signals (n, 2500) and labels (1 = AF, 0 = normal), labels unused until section 3.
    rng = np.random.default_rng(seed)                         # random source for the whole set
    X, y = [], []
    for i in range(n):
        af = bool(rng.random() < 0.5)                         # half AF, half normal
        s = ecg(rng.uniform(55, 100), af, seed=seed * 100000 + i) * rng.uniform(0.7, 1.3)   # random rate and amplitude
        s += 0.02 * rng.standard_normal(s.shape)              # sensor noise
        s += 0.05 * np.sin(2 * np.pi * rng.uniform(0.1, 0.4) * t_axis + rng.uniform(0, 6.28))   # slow baseline wander
        X.append(s); y.append(int(af))
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(y)

X_ecg, y_ecg = make_ecgs(3000, seed=1)                        # pretraining set (labels set aside)
X_ecg_test, y_ecg_test = make_ecgs(1000, seed=2)              # test set
print("pretraining:", tuple(X_ecg.shape), "| test:", tuple(X_ecg_test.shape))

### 1.2 Cut the signal into one-second patches: each patch becomes a token
Ten seconds at 250 samples per second is 2,500 numbers. Cutting it into one-second patches gives **10 tokens of 250 numbers**
each, just as words become tokens for a language model.

In [ ]:
N_PATCH, PATCH_LEN = 10, FS                                   # 10 patches of 250 samples

def ecg_patches(x):
    # (B, 2500) -> (B, 10, 250): one row per one-second patch.
    return x.reshape(x.shape[0], N_PATCH, PATCH_LEN)

print("one recording as tokens:", tuple(ecg_patches(torch.tensor(x, dtype=torch.float32)[None]).shape))

### 1.3 Hide most of the patches at random
The clip hides patches 2, 3, 5, 7, 8, and 10 (indices 1, 2, 4, 6, 7, 9 counting from zero): **6 of 10**. During training,
a new random set of 6 is hidden every time.

In [ ]:
MASK = [1, 2, 4, 6, 7, 9]                                     # hidden patches in clip 1
plt.figure(figsize=(12, 2))
for k in range(N_PATCH):                                      # draw each patch, greyed out if hidden
    seg = slice(k * FS, (k + 1) * FS)
    if k in MASK: plt.axvspan(k, k + 1, color="0.85")         # grey box for a hidden patch
    else: plt.plot(t_axis[seg], x[seg], lw=1, color="k")      # visible patch
plt.title("6 of 10 patches hidden"); plt.xlabel("seconds"); plt.show()

### 1.4 The encoder sees only the visible patches
The encoder is a small transformer. A linear layer turns each patch into a vector of **256 numbers**, a learned position
vector is added, and two transformer layers mix the tokens. When `keep` is given, only those patches go in: here, 4 of 10.
`embed_all` averages the output vectors into one embedding per recording, which we use in sections 2 and 3.

The same `Encoder` class is used for every encoder in this notebook (masked, contrastive, and the random baselines), so all
comparisons are between identical architectures.

In [ ]:
D = 256                                                       # embedding size, as in clip 3

class Encoder(nn.Module):
    def __init__(self, patch_dim, n_patches, d=D, depth=2, heads=4):
        super().__init__()
        self.embed = nn.Linear(patch_dim, d)                  # patch -> token vector
        self.pos = nn.Parameter(torch.randn(1, n_patches, d) * 0.02)   # learned position vectors
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)   # two transformer layers
        self.norm = nn.LayerNorm(d)                           # final normalization

    def forward(self, patches, keep=None):
        # patches (B, n, patch_dim) -> token vectors (B, n_kept, d); keep = indices of the visible patches, or None for all.
        x = self.embed(patches) + self.pos                    # tokens with positions
        if keep is not None:                                  # drop the hidden patches before the transformer
            x = torch.gather(x, 1, keep[..., None].expand(-1, -1, x.shape[-1]))
        return self.norm(self.blocks(x))

    def embed_all(self, patches):
        # One embedding per input: the mean of all token vectors, shape (B, d).
        return self.forward(patches).mean(dim=1)

enc_demo = Encoder(PATCH_LEN, N_PATCH)                        # an untrained encoder, just to check shapes
keep = torch.tensor([[k for k in range(N_PATCH) if k not in MASK]])   # the 4 visible patches
print("visible patches:", keep.tolist()[0])
print("encoder output:", tuple(enc_demo(ecg_patches(torch.tensor(x, dtype=torch.float32)[None]), keep).shape), "= 4 tokens of 256 numbers")

### 1.5 A small decoder predicts the missing waveform from the encoder's output
The decoder receives the 4 encoded tokens back in their places, plus a learned **mask token** in each of the 6 hidden places.
One transformer layer and a linear layer turn every token back into 250 samples.

### 1.6 The loss compares prediction and truth only on the hidden patches
Mean squared error, computed only where the patches were hidden. The visible patches are not scored: copying them would be
too easy.

In [ ]:
class MAE(nn.Module):
    def __init__(self, patch_dim, n_patches, n_hidden, d=D):
        super().__init__()
        self.n_patches, self.n_hidden = n_patches, n_hidden
        self.encoder = Encoder(patch_dim, n_patches, d)       # kept after pretraining
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d))  # stands in for every hidden patch
        self.dec_pos = nn.Parameter(torch.randn(1, n_patches, d) * 0.02)   # decoder position vectors
        layer = nn.TransformerEncoderLayer(d, 4, 2 * d, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)   # deliberately small
        self.out = nn.Linear(d, patch_dim)                    # token -> patch values

    def forward(self, patches):
        B = patches.shape[0]
        order = torch.rand(B, self.n_patches).argsort(dim=1)  # a random order of patches for each input
        keep, hide = order[:, : self.n_patches - self.n_hidden], order[:, self.n_patches - self.n_hidden:]
        z = self.encoder(patches, keep)                       # encode the visible patches only
        full = self.mask_token.expand(B, self.n_patches, -1).clone()   # mask tokens everywhere ...
        full.scatter_(1, keep[..., None].expand(-1, -1, z.shape[-1]), z)   # ... then the encoded visible patches in their places
        pred = self.out(self.decoder(full + self.dec_pos))    # predicted values for every patch
        pick = lambda a: torch.gather(a, 1, hide[..., None].expand(-1, -1, a.shape[-1]))   # select the hidden patches
        loss = F.mse_loss(pick(pred), pick(patches))          # loss on hidden patches only
        return loss, pred, hide

### 1.7 To fill the gaps, the encoder has to learn what a heartbeat looks like
We train on the 3,000 unlabeled recordings for 15 passes. Watch the loss, then look at the predictions for a test recording.

In [ ]:
def pretrain_mae(model, X, patchify, epochs=15, batch=128):
    # Masked-reconstruction pretraining on unlabeled inputs X; prints the mean loss of each pass.
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    for ep in range(epochs):
        perm, total = torch.randperm(len(X)), 0.0             # shuffle
        for i in range(0, len(X), batch):
            loss, _, _ = model(patchify(X[perm[i:i + batch]]))   # no labels anywhere
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(perm[i:i + batch])
        if ep in (0, 4, 9, epochs - 1): print(f"pass {ep + 1:>2}: reconstruction loss {total / len(X):.4f}")
    return model

t0 = time.time()
torch.manual_seed(0); mae_ecg = pretrain_mae(MAE(PATCH_LEN, N_PATCH, n_hidden=6), X_ecg, ecg_patches)
print(f"{time.time() - t0:.0f} s")

In [ ]:
mae_ecg.eval()
torch.manual_seed(1)
with torch.no_grad():
    _, pred, hide = mae_ecg(ecg_patches(X_ecg_test[:2]))      # predictions for two test recordings
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
for r in range(2):
    axes[r].plot(t_axis, X_ecg_test[r], lw=1, color="k", label="true signal")
    for k in hide[r].tolist():                                # overlay the prediction on each hidden patch
        seg = slice(k * FS, (k + 1) * FS)
        axes[r].axvspan(k, k + 1, color="0.92")
        axes[r].plot(t_axis[seg], pred[r, k], color="tab:red", lw=1.5)
    axes[r].set_title(("AF" if y_ecg_test[r] else "normal") + " test recording: hidden patches shaded, prediction in red", fontsize=9)
plt.xlabel("seconds"); plt.tight_layout(); plt.show()

**What to look for.** The loss drops quickly and then stays nearly flat. The red predictions are close to a flat line at the
average value of the signal, not heartbeats. This tiny masked autoencoder learned little about heartbeats: the exact time of each beat in a hidden second
is hard to predict, and the mean squared error rewards a blurred average. Real ECG foundation models use much larger
encoders, far more data, and longer training. We will see in section 3 what this weak result means for the encoder.

### 1.8 After pretraining, keep the encoder and discard the decoder

In [ ]:
encoder_mae_ecg = mae_ecg.encoder                             # keep the encoder; the decoder is not used again
print("encoder weights:", sum(p.numel() for p in encoder_mae_ecg.parameters()))

### 1.9 The same recipe works for pathology: hide image patches, reconstruct the pixels
We download the Kather 2016 tiles and shrink each 150 × 150 tile to 64 × 64 so training stays fast on a CPU. A 4 × 4 grid
gives **16 patches** of 16 × 16 pixels, and we hide **12 of 16**, as in the clip. 4,000 tiles are used for pretraining
(without labels) and 1,000 are held out for testing.

In [ ]:
URL = "https://zenodo.org/api/records/53169/files/Kather_texture_2016_image_tiles_5000.zip/content"
zip_path = os.path.join(DATA_DIR, "Kather_texture_2016_image_tiles_5000.zip")
for attempt in range(3):                                      # download once (about 260 MB), retrying if the connection drops
    if os.path.exists(zip_path): break
    try:
        print("downloading the Kather 2016 tiles ...")
        urllib.request.urlretrieve(URL, zip_path + ".part")   # write to a temporary name first ...
        os.replace(zip_path + ".part", zip_path)              # ... so a broken download is never mistaken for the zip
    except OSError as error:
        print("download failed:", error, "| retrying" if attempt < 2 else "| check the connection and run this cell again")
tile_dir = os.path.join(DATA_DIR, "Kather_texture_2016_image_tiles_5000")
if not os.path.isdir(tile_dir):                               # unzip once
    with zipfile.ZipFile(zip_path) as z: z.extractall(DATA_DIR)

CLASSES = sorted(os.listdir(tile_dir))                        # 01_TUMOR, 02_STROMA, ..., 08_EMPTY
PX = 64                                                       # tile size after resizing
images, labels = [], []
for k, c in enumerate(CLASSES):                               # read every tile of every class
    for f in sorted(glob.glob(os.path.join(tile_dir, c, "*.tif"))):
        images.append(np.asarray(Image.open(f).convert("RGB").resize((PX, PX), Image.BILINEAR)))
        labels.append(k)
X_he = torch.tensor(np.array(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255   # (5000, 3, 64, 64) in [0, 1]
y_he = torch.tensor(labels)                                   # class index, unused until section 3
perm = torch.randperm(len(X_he), generator=torch.Generator().manual_seed(0))
he_train, he_test = perm[:4000], perm[4000:]                  # 4,000 for pretraining, 1,000 held out
print("tiles:", tuple(X_he.shape), "| classes:", [c[3:].lower() for c in CLASSES])

In [ ]:
G, P = 4, PX // 4                                             # 4 x 4 grid of 16 x 16 patches

def he_patches(x):
    # (B, 3, 64, 64) -> (B, 16, 768): one row per patch, left to right, top to bottom.
    B = x.shape[0]
    p = x.unfold(2, P, P).unfold(3, P, P)                     # (B, 3, 4, 4, 16, 16)
    return p.permute(0, 2, 3, 1, 4, 5).reshape(B, G * G, 3 * P * P)

def he_unpatch(p):
    # The inverse of he_patches, for display.
    B = p.shape[0]
    return p.reshape(B, G, G, 3, P, P).permute(0, 3, 1, 4, 2, 5).reshape(B, 3, PX, PX)

t0 = time.time()
torch.manual_seed(0); mae_he = pretrain_mae(MAE(3 * P * P, G * G, n_hidden=12), X_he[he_train], he_patches)
print(f"{time.time() - t0:.0f} s")

In [ ]:
mae_he.eval()
torch.manual_seed(2)
with torch.no_grad():
    imgs = X_he[he_test[:6]]                                  # six held-out tiles
    _, pred, hide = mae_he(he_patches(imgs))
    shown, masked = he_patches(imgs).clone(), he_patches(imgs).clone()
    for i in range(len(imgs)):
        shown[i, hide[i]] = pred[i, hide[i]]                  # paste the predictions into the hidden places
        masked[i, hide[i]] = 0.85                             # grey for display
fig, axes = plt.subplots(3, 6, figsize=(10, 5.2))
for j in range(6):
    for r, (im, title) in enumerate([(imgs[j], "tile"), (he_unpatch(masked)[j], "12 of 16 hidden"), (he_unpatch(shown)[j], "reconstruction")]):
        axes[r, j].imshow(im.clamp(0, 1).permute(1, 2, 0)); axes[r, j].axis("off")
        if j == 0: axes[r, j].set_title(title, fontsize=9, loc="left")
plt.suptitle("H&E tiles: Kather et al. 2016, CC BY 4.0", fontsize=8); plt.tight_layout(); plt.show()
encoder_mae_he = mae_he.encoder                               # keep the encoder, discard the decoder

**What to look for.** The hidden patches are filled mostly with the average color of each tile: blurry, like the ECG
predictions. Even so, to get the color right from 4 visible patches, the encoder must summarize the stain and tissue it sees.
Section 3 measures whether that summary is useful.

## 2. Contrastive learning: two views, one embedding  *(clip 2)*

### 2.1 Make two augmented views of the same ECG
### 2.2 Augmentations change the recording, not the diagnosis
As in the clip, a view can be shifted in time (a different crop of the recording), rescaled, given baseline wander, or given
extra noise. None of this changes the rhythm, so it does not change the diagnosis.

In [ ]:
def augment_ecg(x):
    # Random view of each recording in x (B, 2500): time shift, scaling, baseline wander, noise.
    B, n = x.shape
    t = torch.arange(n) / FS
    out = x * (0.6 + 0.8 * torch.rand(B, 1))                  # scaling
    out = out + 0.2 * torch.rand(B, 1) * torch.sin(2 * np.pi * (0.1 + 0.4 * torch.rand(B, 1)) * t + 6.28 * torch.rand(B, 1))   # baseline wander
    out = out + 0.05 * torch.rand(B, 1) * torch.randn(B, n)   # noise
    shift = torch.randint(0, n, (B,))                         # time shift, like a different crop
    return torch.stack([torch.roll(out[i], int(shift[i])) for i in range(B)])

torch.manual_seed(3)
a = X_ecg[:1]                                                 # ECG A
fig, axes = plt.subplots(3, 1, figsize=(12, 4.5), sharex=True)
for ax, (sig, name) in zip(axes, [(a, "ECG A"), (augment_ecg(a), "view 1"), (augment_ecg(a), "view 2")]):
    ax.plot(t_axis, sig[0], lw=1); ax.set_title(name, fontsize=9, loc="left")
plt.xlabel("seconds"); plt.tight_layout(); plt.show()

### 2.3 One shared encoder maps each view to a point in embedding space
### 2.4 The other ECGs in the batch add more points
Both views go through the **same** encoder (shared weights), and every other recording in the batch adds two more points.

### 2.5 Training pulls the two views together and pushes the other ECGs apart
### 2.6 The loss asks: which view in the batch is my partner?
The clip compares view A1 with A2 (its partner) and with B, C, and D. Their similarities to A1 are 0.9, 0.2, −0.1, and 0.3.
Dividing by the temperature τ = 0.5 and applying a softmax gives the probabilities in the clip:

In [ ]:
similarities = torch.tensor([0.9, 0.2, -0.1, 0.3])            # similarity of A1 to A2 (partner), B, C, D
tau = 0.5                                                     # temperature
probabilities = torch.softmax(similarities / tau, dim=0)      # which one is the partner?
print("probabilities:", probabilities.numpy().round(2))      # 0.59 for the partner A2, as in the clip
print("loss for A1 = -log p(A2) =", round(-torch.log(probabilities[0]).item(), 2))

### 2.7 This is the InfoNCE loss: a softmax over similarities with temperature τ
In a batch of N recordings there are 2N views. For each view, its partner is the correct answer and the other 2N − 2 views
are the wrong ones. As in SimCLR, a small **projection head** sits on top of the encoder during training and is discarded
afterwards, like the masked autoencoder's decoder.

In [ ]:
def info_nce(z1, z2, tau=0.5):
    # InfoNCE loss; row i of z1 and row i of z2 are two views of the same input.
    z = F.normalize(torch.cat([z1, z2]), dim=1)               # 2N unit vectors
    sim = z @ z.T / tau                                       # cosine similarities / temperature
    sim.fill_diagonal_(float("-inf"))                         # a view is not its own partner
    N = z1.shape[0]
    partner = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)])   # index of each view's partner
    return F.cross_entropy(sim, partner)                      # softmax over each row, -log p(partner)

def pretrain_contrastive(encoder, X, patchify, augment, epochs=15, batch=256):
    # Contrastive pretraining on unlabeled inputs X; returns the encoder (the projection head is discarded).
    head = nn.Sequential(nn.Linear(D, D), nn.ReLU(), nn.Linear(D, 64))   # projection head
    opt = torch.optim.AdamW(list(encoder.parameters()) + list(head.parameters()), lr=1e-3)
    for ep in range(epochs):
        perm, total = torch.randperm(len(X)), 0.0
        for i in range(0, len(X), batch):
            xb = X[perm[i:i + batch]]                         # no labels
            z1 = head(encoder.embed_all(patchify(augment(xb))))   # view 1
            z2 = head(encoder.embed_all(patchify(augment(xb))))   # view 2, same encoder
            loss = info_nce(z1, z2)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(xb)
        if ep in (0, 4, 9, epochs - 1): print(f"pass {ep + 1:>2}: InfoNCE loss {total / len(X):.3f}")
    return encoder

t0 = time.time()
torch.manual_seed(0); encoder_cl_ecg = pretrain_contrastive(Encoder(PATCH_LEN, N_PATCH), X_ecg, ecg_patches, augment_ecg)
print(f"{time.time() - t0:.0f} s")

### 2.8 Without any labels, similar ECGs tend to end up close together
We embed the test recordings and project the 256 numbers onto two dimensions (PCA). The colors use the labels, which the
encoder never saw.

In [ ]:
encoder_cl_ecg.eval()
with torch.no_grad(): z = encoder_cl_ecg.embed_all(ecg_patches(X_ecg_test))   # (1000, 256)
z = z - z.mean(0)                                             # center
_, _, V = torch.pca_lowrank(z, q=2, center=False)             # top two directions
xy = z @ V                                                    # 2D coordinates
plt.figure(figsize=(4.5, 4))
for lab, name, col in [(0, "normal sinus rhythm", "#0F766E"), (1, "atrial fibrillation", "#7C3AED")]:
    m = y_ecg_test == lab
    plt.scatter(xy[m, 0], xy[m, 1], s=6, color=col, label=name)
plt.legend(fontsize=8); plt.title("contrastive embeddings (PCA)", fontsize=10); plt.show()

### 2.9 For pathology tiles, views come from crops, flips, and stain color changes
Tissue has no fixed orientation, so rotations and flips are safe. Stain color varies between labs, so we change it a little.

In [ ]:
def augment_he(x):
    # Random view of each tile: crop and resize, 90-degree rotation, flip, small stain color change.
    out = []
    for im in x:
        s = int(torch.randint(40, PX + 1, (1,)))              # crop size
        a, b = torch.randint(0, PX - s + 1, (2,))             # crop corner
        im = F.interpolate(im[None, :, a:a + s, b:b + s], size=PX, mode="bilinear", align_corners=False)[0]
        im = torch.rot90(im, int(torch.randint(0, 4, (1,))), dims=(1, 2))   # rotation
        if torch.rand(1) < 0.5: im = im.flip(2)               # flip
        im = im * (1 + 0.16 * (torch.rand(3, 1, 1) - 0.5)) + 0.08 * (torch.rand(3, 1, 1) - 0.5)   # stain color change
        out.append(im.clamp(0, 1))
    return torch.stack(out)

torch.manual_seed(4)
tile = X_he[he_train[:1]]
fig, axes = plt.subplots(1, 3, figsize=(6, 2.2))
for ax, (im, name) in zip(axes, [(tile[0], "tile"), (augment_he(tile)[0], "view 1"), (augment_he(tile)[0], "view 2")]):
    ax.imshow(im.permute(1, 2, 0)); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

t0 = time.time()
torch.manual_seed(0); encoder_cl_he = pretrain_contrastive(Encoder(3 * P * P, G * G), X_he[he_train], he_patches, augment_he)
print(f"{time.time() - t0:.0f} s")

## 3. Linear probing: what did the encoder learn?  *(clip 3)*

### 3.1 Take the pretrained encoder and freeze its weights

In [ ]:
def freeze(encoder):
    # No weight of the encoder can change: no gradients are computed for it.
    for p in encoder.parameters():
        p.requires_grad_(False)                               # no gradient from now on
        p.grad = None                                         # forget gradients left over from pretraining
    return encoder.eval()

freeze(encoder_cl_ecg)
print("trainable encoder weights:", sum(p.numel() for p in encoder_cl_ecg.parameters() if p.requires_grad))

### 3.2 Now use a small labeled set: atrial fibrillation versus normal sinus rhythm
We take only **10 labeled recordings per class** from the pretraining set.

### 3.3 The frozen encoder turns each ECG into a 256-dimensional embedding

In [ ]:
def small_labeled_set(y, n_per_class, seed=0):
    # Indices of n_per_class random examples of each class (0 and 1).
    g = torch.Generator().manual_seed(seed)
    return torch.cat([torch.nonzero(y == c)[:, 0][torch.randperm(int((y == c).sum()), generator=g)[:n_per_class]] for c in (0, 1)])

idx = small_labeled_set(y_ecg, 10)                            # 20 labeled recordings
with torch.no_grad():
    z_train = encoder_cl_ecg.embed_all(ecg_patches(X_ecg[idx]))   # (20, 256)
print("labeled recordings:", len(idx), "| embedding of each:", z_train.shape[1], "numbers")

### 3.4 Add one linear prediction head: a weighted sum of the embedding plus a bias
p(AF) = σ(w · z + b): **256 weights and 1 bias**.

### 3.5 Training updates only the head; the encoder gets no gradient
We standardize each of the 256 features (using the labeled set), then train the head with the binary cross-entropy loss.

In [ ]:
def linear_probe(encoder, X_tr, y_tr, X_te, y_te, patchify, n_per_class, seed=0, show=False):
    # Train a linear head on a frozen encoder with n_per_class labels per class; return test accuracy.
    freeze(encoder)
    with torch.no_grad():
        f_te = encoder.embed_all(patchify(X_te))              # test embeddings
        idx = small_labeled_set(y_tr, n_per_class, seed)
        f_tr = encoder.embed_all(patchify(X_tr[idx]))         # labeled embeddings
    mu, sd = f_tr.mean(0), f_tr.std(0) + 1e-6                 # standardize each feature
    torch.manual_seed(seed)
    head = nn.Linear(D, 1)                                    # w (256) and b (1)
    if show: print("trainable head weights:", sum(p.numel() for p in head.parameters()))
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-3)
    for _ in range(300):
        loss = F.binary_cross_entropy_with_logits(head((f_tr - mu) / sd)[:, 0], y_tr[idx].float())
        opt.zero_grad(); loss.backward(); opt.step()          # gradients reach the head only
    if show: print("encoder received a gradient:", any(p.grad is not None for p in encoder.parameters()))   # False
    with torch.no_grad():
        return ((head((f_te - mu) / sd)[:, 0] > 0).long() == y_te).float().mean().item()

acc = linear_probe(encoder_cl_ecg, X_ecg, y_ecg, X_ecg_test, y_ecg_test, ecg_patches, n_per_class=10, show=True)
print(f"AF versus normal, 10 labels per class, contrastive encoder: test accuracy {acc:.3f}")

### 3.6 If one line separates the classes, the information is already in the embedding
A linear head can only draw a straight boundary (a hyperplane) in the 256-dimensional space. So high probe accuracy means
the pretrained encoder already arranged AF and normal recordings on two sides.

To know whether pretraining helped, we need a **baseline**: the same `Encoder` architecture with random weights, never
trained. We probe all three encoders with 10 and 100 labels per class, averaging 5 random draws of the labeled set.

In [ ]:
def table(rows, fn, sizes=(10, 100), seeds=5):
    # Mean test accuracy of fn(encoder, n_per_class, seed) for each encoder and labeled-set size.
    print(f"{'':>22s}" + "".join(f"{n:>4d} labels/class" for n in sizes))
    out = {}
    for name, enc in rows.items():
        out[name] = [float(np.mean([fn(enc, n, s) for s in range(seeds)])) for n in sizes]
        print(f"{name:>22s}" + "".join(f"{a:>17.3f}" for a in out[name]))
    return out

torch.manual_seed(1); encoder_rand_ecg = Encoder(PATCH_LEN, N_PATCH)   # random baseline, same architecture
ecg_encoders = {"random (baseline)": encoder_rand_ecg, "masked reconstruction": encoder_mae_ecg, "contrastive": encoder_cl_ecg}
probe_ecg = table(ecg_encoders, lambda e, n, s: linear_probe(e, X_ecg, y_ecg, X_ecg_test, y_ecg_test, ecg_patches, n, s))

**Reading the table.** Chance is 0.5. The contrastive encoder is far ahead of the random baseline: its embedding already
separates AF from normal rhythm, so a single line reads it out. The masked-reconstruction encoder is barely above the random
baseline. That fits what we saw in section 1.7: this tiny model learned little from reconstruction.

### 3.7 Fine-tuning updates the encoder too, so a probe is the cleaner test of pretraining
With fine-tuning, every weight is trainable: the encoder starts from its pretrained weights and changes together with the head.
That can reach higher accuracy, but it no longer measures only what pretraining learned, because the labels also shape the
encoder. We fine-tune a copy of each encoder for 150 steps and average 3 random draws of the labeled set.

Compare the table below with the probe table above. With only 10 labels per class, fine-tuning the contrastive encoder does
**worse** than probing it: a million trainable weights and 20 examples is an easy way to overfit. Fine-tuning pays off when
there are more labels.

In [ ]:
def fine_tune(encoder, X_tr, y_tr, X_te, y_te, patchify, n_per_class, seed=0, steps=150):
    # Train a copy of the encoder and a linear head together on n_per_class labels per class; return test accuracy.
    enc = copy.deepcopy(encoder).train()                      # a copy, so the pretrained encoder stays unchanged
    for p in enc.parameters(): p.requires_grad_(True)         # every weight is trainable
    idx = small_labeled_set(y_tr, n_per_class, seed)
    torch.manual_seed(seed)
    head = nn.Linear(D, 1)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(head.parameters()), lr=1e-4)
    for _ in range(steps):
        loss = F.binary_cross_entropy_with_logits(head(enc.embed_all(patchify(X_tr[idx])))[:, 0], y_tr[idx].float())
        opt.zero_grad(); loss.backward(); opt.step()
    enc.eval()
    with torch.no_grad():
        return ((head(enc.embed_all(patchify(X_te)))[:, 0] > 0).long() == y_te).float().mean().item()

n_all = sum(p.numel() for p in encoder_cl_ecg.parameters()) + D + 1
print(f"trainable weights: linear probe {D + 1}, fine-tuning {n_all:,}")
t0 = time.time()
ft_ecg = table(ecg_encoders, lambda e, n, s: fine_tune(e, X_ecg, y_ecg, X_ecg_test, y_ecg_test, ecg_patches, n, s), seeds=3)
print(f"{time.time() - t0:.0f} s")

### 3.8 The same probe works for pathology: tumor versus normal mucosa
We keep only two of the eight classes, tumor and normal mucosa, and run the same probe and fine-tuning on the three H&E
encoders: random baseline, masked reconstruction, and contrastive.

In [ ]:
def two_classes(idx):
    # Tiles of tumor (label 1) and normal mucosa (label 0) among the given indices.
    tumor, mucosa = CLASSES.index("01_TUMOR"), CLASSES.index("06_MUCOSA")
    keep = idx[(y_he[idx] == tumor) | (y_he[idx] == mucosa)]
    return X_he[keep], (y_he[keep] == tumor).long()

Xb_tr, yb_tr = two_classes(he_train)                          # pool for the small labeled sets
Xb_te, yb_te = two_classes(he_test)                           # held-out tiles
print("tumor + mucosa tiles: pool", len(yb_tr), "| test", len(yb_te))

torch.manual_seed(1); encoder_rand_he = Encoder(3 * P * P, G * G)   # random baseline, same architecture
he_encoders = {"random (baseline)": encoder_rand_he, "masked reconstruction": encoder_mae_he, "contrastive": encoder_cl_he}
print("\nlinear probe")
probe_he = table(he_encoders, lambda e, n, s: linear_probe(e, Xb_tr, yb_tr, Xb_te, yb_te, he_patches, n, s))
print("\nfine-tuning")
ft_he = table(he_encoders, lambda e, n, s: fine_tune(e, Xb_tr, yb_tr, Xb_te, yb_te, he_patches, n, s), seeds=3)

**Reading the tables.** On the H&E tiles, both pretrained encoders beat the random baseline under the linear probe, and the
masked-reconstruction and contrastive encoders are close to each other. After fine-tuning, the masked-reconstruction encoder
is the best of the three with both label counts. He et al. (MAE, the Thursday paper) report the same pattern at scale: MAE
features are weaker than contrastive features under a linear probe, because pixel reconstruction keeps detail that a single
linear layer cannot use directly, but they are among the strongest once the encoder is fine-tuned. On the ECGs, our
masked-reconstruction encoder learned too little in section 1.7 to show this pattern.

Your exact numbers can differ slightly from run to run and between CPU and GPU. With a few labels, the spread between random
draws of the labeled set is a few points.

## 4. Optional: features from a pretrained imaging foundation model
Pathology foundation models such as UNI (the Thursday paper) are gated on Hugging Face: you must request access and agree to
the terms. Here we use **DINOv2-small**, a general-purpose model trained with the same self-distillation recipe that UNI uses,
on natural images rather than pathology. We give it the original 150 × 150 tiles resized to 224 × 224, take its `[CLS]` vector
(384 numbers), and run the same tumor versus mucosa probe. Set `RUN_DINOV2 = False` to skip.

In [ ]:
RUN_DINOV2 = True
if RUN_DINOV2:
    try:
        from transformers import AutoModel                    # pip install transformers if this fails
        dino = AutoModel.from_pretrained("facebook/dinov2-small").eval()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)   # ImageNet normalization used by DINOv2
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        class Dino(nn.Module):                                # wraps DINOv2 so linear_probe can use it
            def embed_all(self, x):
                x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
                return dino(pixel_values=(x - mean) / std).last_hidden_state[:, 0]   # [CLS] vector
        D_saved, D = D, 384                                   # the probe head now has 384 inputs
        acc = [linear_probe(Dino(), Xb_tr, yb_tr, Xb_te, yb_te, lambda x: x, n) for n in (10, 100)]
        D = D_saved
        print(f"DINOv2-small linear probe, tumor versus mucosa: 10 labels/class {acc[0]:.3f}, 100 labels/class {acc[1]:.3f}")
    except Exception as error:
        print("Skipped the DINOv2 section:", type(error).__name__, error)

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. Masked reconstruction | `ecg`, `ecg_patches`, `Encoder`, `MAE` (6 of 10 ECG patches, 12 of 16 tile patches hidden) |
| 2. Contrastive learning | `augment_ecg`, `augment_he`, `info_nce`, `pretrain_contrastive` |
| 3. Linear probing | `freeze`, `linear_probe` (257 trainable weights), `fine_tune`, random baselines with the same architecture |

> **Try it (optional):** hide 3 or 8 ECG patches instead of 6 and compare the probe. Train the ECG masked autoencoder for
> 60 passes instead of 15. Remove the rotation from `augment_he`, or make the stain change much stronger: does the probe drop?